In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip uninstall -y torchao -q
!pip install -q transformers accelerate datasets evalplus sentencepiece
print("Done")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 4.6 MB/s eta 0:00:00
Done


In [3]:
import os, gc, json, sys, subprocess, tempfile, time, traceback, warnings
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
from collections import OrderedDict
warnings.filterwarnings("ignore")

import torch
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GPU 0: Tesla T4 (15.6 GB)
GPU 1: Tesla T4 (15.6 GB)


In [4]:
MODEL_ID   = "sandeeprdy1729/TIMPS-Coder-7B"
OUTPUT_DIR = "/kaggle/working/timps-benchmarks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.1
TOP_P          = 0.95

from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {MODEL_ID} ...")
t0 = time.time()
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=dtype, device_map="auto", trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
print(f"Loaded in {time.time()-t0:.1f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading sandeeprdy1729/TIMPS-Coder-7B ...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loaded in 95.5s | VRAM: 6.68 GB


In [9]:
def extract_code(generation: str) -> str:
    text = generation.strip()
    if "```python" in text:
        parts = text.split("```python")
        if len(parts) > 1:
            return parts[1].split("```")[0].strip()
    if "```" in text:
        parts = text.split("```")
        for part in parts[1:]:
            code = part.strip()
            first = code.split("\n")[0].strip()
            if first and not first.startswith(("def ", "class ", "import ", "from ")):
                code = "\n".join(code.split("\n")[1:]).strip()
            if code:
                return code
    lines = text.split("\n")
    code_lines, started = [], False
    for line in lines:
        s = line.strip()
        if s.startswith(("def ", "class ", "import ", "from ")):
            started = True
        if started:
            code_lines.append(line)
    return "\n".join(code_lines).strip() if code_lines else text


def run_test(code: str, test_code: str, timeout: float = 10.0) -> Dict[str, Any]:
    full = code + "\n\n" + test_code + "\n\nprint('TEST_PASSED')\n"
    tmp = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(full); tmp = f.name
        proc = subprocess.run(
            [sys.executable, tmp], capture_output=True, text=True, timeout=timeout,
            env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
        )
        passed = "TEST_PASSED" in proc.stdout
        return {"passed": passed, "error": None if passed else (proc.stderr[-300:] or "tests did not pass")}
    except subprocess.TimeoutExpired:
        return {"passed": False, "error": "timeout"}
    except Exception as e:
        return {"passed": False, "error": str(e)[:200]}
    finally:
        if tmp:
            try: os.unlink(tmp)
            except OSError: pass


def generate_one(prompt: str) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": "You are an expert Python programmer. Write a complete, correct, and efficient solution. Output ONLY the code, no explanation."},
            {"role": "user", "content": prompt},
        ]
        try:
            chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            chat = f"<|im_start|>system\nYou are an expert Python programmer. Write a complete, correct, and efficient solution. Output ONLY the code, no explanation.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    else:
        chat = f"<|im_start|>system\nYou are an expert Python programmer. Write a complete, correct, and efficient solution. Output ONLY the code, no explanation.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(chat, return_tensors="pt", truncation=True, max_length=1536)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
            top_p=TOP_P, do_sample=(TEMPERATURE > 0),
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
from datasets import load_dataset

print("=" * 60)
print("  BENCHMARK: HumanEval  (164 problems)")
print("=" * 60)

ds = load_dataset("openai/openai_humaneval", split="test")
problems = list(ds)
total = len(problems)
passed = failed = errors = 0
he_details = []

t0 = time.time()
for i, prob in enumerate(problems):
    prompt, test, entry_point = prob["prompt"], prob["test"], prob["entry_point"]
    try:
        gen = generate_one(prompt)
        code = extract_code(gen)
        res = run_test(code, test)
        ok = res["passed"]
    except Exception as e:
        ok, res, code = False, {"passed": False, "error": str(e)[:200]}, ""

    if ok: passed += 1; tag = "PASS"
    elif res.get("error") and ("Error" in str(res["error"]) or "timeout" in str(res.get("error","")).lower()):
        errors += 1; tag = "ERR"
    else: failed += 1; tag = "FAIL"

    # NOTE: we now store the generated `code` so HumanEval+ can reuse it
    # instead of re-running generation for all 164 problems a second time.
    he_details.append({"idx": i, "task_id": prob["task_id"], "entry_point": entry_point,
                        "passed": ok, "tag": tag, "error": res.get("error"), "code": code})
    if (i + 1) % 10 == 0 or i == total - 1:
        print(f"  [{i+1:>3}/{total}]  {tag:<5}  running pass@1 = {passed/(i+1)*100:.1f}%")

he_result = {"benchmark": "HumanEval", "total": total, "passed": passed, "failed": failed,
             "errors": errors, "pass@1": round(passed/total*100, 1), "per_problem": he_details}
print(f"\n  HumanEval done in {(time.time()-t0)/60:.1f} min  |  pass@1 = {he_result['pass@1']}%")

In [ ]:
print("=" * 60)
print("  BENCHMARK: HumanEval+  (extended tests via evalplus)")
print("=" * 60)

from evalplus.data import get_human_eval_plus
from evalplus.eval import PASS

t0 = time.time()
hep_result = None

try:
    hep_problems = get_human_eval_plus()  # same 164 problems, with extra "plus" tests

    # Reuse the code already generated during the HumanEval pass above —
    # no need to regenerate (this alone halves the GPU time this cell used to burn).
    samples_path = f"{OUTPUT_DIR}/humaneval_plus_samples.jsonl"
    result_path = samples_path.replace(".jsonl", "_eval_results.json")
    for p in (samples_path, result_path):
        if os.path.exists(p):
            os.remove(p)  # evalplus prompts interactively if a stale result file exists

    covered = set()
    with open(samples_path, "w") as f:
        for d in he_details:
            task_id = d["task_id"]
            if task_id in hep_problems and d["code"]:
                f.write(json.dumps({"task_id": task_id, "solution": d["code"]}) + "\n")
                covered.add(task_id)

    missing = set(hep_problems) - covered
    if missing:
        print(f"  Filling {len(missing)} problem(s) missing/empty from the base run ...")
        with open(samples_path, "a") as f:
            for task_id in missing:
                try:
                    gen = generate_one(hep_problems[task_id]["prompt"])
                    code = extract_code(gen)
                except Exception:
                    code = ""
                f.write(json.dumps({"task_id": task_id, "solution": code}) + "\n")

    from evalplus.evaluate import evaluate as ev_eval
    ev_eval(dataset="humaneval", samples=samples_path, i_just_wanna_run=True)

    with open(result_path) as f:
        raw = json.load(f)
    n = len(raw["eval"])
    base_pass = sum(1 for r in raw["eval"].values() if r[0]["base_status"] == PASS)
    plus_pass = sum(1 for r in raw["eval"].values()
                     if r[0]["base_status"] == PASS and r[0]["plus_status"] == PASS)
    hep_result = {
        "benchmark": "HumanEval+", "total": n, "passed": plus_pass,
        "pass@1": round(plus_pass / n * 100, 1),
        "base_pass@1": round(base_pass / n * 100, 1),
        "evalplus_raw_path": result_path,
    }
    print(f"  HumanEval+ pass@1 = {hep_result['pass@1']}%  (base-only pass@1 = {hep_result['base_pass@1']}%)")

except Exception as e:
    print(f"[humaneval+] evalplus failed: {e}")
    traceback.print_exc()
    passed = sum(1 for d in he_details if d["passed"])
    hep_result = {"benchmark": "HumanEval+", "total": len(he_details), "passed": passed,
                  "pass@1": round(passed/len(he_details)*100, 1), "note": f"evalplus error: {e}",
                  "per_problem": he_details}

print(f"  Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
print("=" * 60)
print("  BENCHMARK: MBPP  (974 problems)")
print("=" * 60)

ds_mbpp = load_dataset("google-research-datasets/mbpp", split="test")
mbpp_problems = list(ds_mbpp)
total = len(mbpp_problems)
passed = failed = errors = 0
mbpp_details = []

t0 = time.time()
for i, prob in enumerate(mbpp_problems):
    # BUG FIX: this dataset's field is "text", not "prompt" (that key doesn't exist
    # and was throwing KeyError: 'prompt' before generating a single sample).
    prompt = prob.get("text", prob.get("prompt", ""))
    test_list = prob.get("test_list", [])
    test_code = "\n".join(test_list) if test_list else ""
    try:
        gen = generate_one(prompt)
        code = extract_code(gen)
        res = run_test(code, test_code)
        ok = res["passed"]
    except Exception as e:
        ok, res, code = False, {"passed": False, "error": str(e)[:200]}, ""

    if ok: passed += 1; tag = "PASS"
    elif res.get("error") and ("Error" in str(res["error"]) or "timeout" in str(res.get("error","")).lower()):
        errors += 1; tag = "ERR"
    else: failed += 1; tag = "FAIL"

    mbpp_details.append({"idx": i, "task_id": prob.get("task_id", f"mbpp_{i}"),
                         "passed": ok, "tag": tag, "error": res.get("error"), "code": code})
    if (i + 1) % 50 == 0 or i == total - 1:
        print(f"  [{i+1:>3}/{total}]  {tag:<5}  running pass@1 = {passed/(i+1)*100:.1f}%")

mbpp_result = {"benchmark": "MBPP", "total": total, "passed": passed, "failed": failed,
               "errors": errors, "pass@1": round(passed/total*100, 1), "per_problem": mbpp_details}
print(f"\n  MBPP done in {(time.time()-t0)/60:.1f} min  |  pass@1 = {mbpp_result['pass@1']}%")

In [ ]:
print("=" * 60)
print("  BENCHMARK: MBPP+  (extended tests via evalplus)")
print("=" * 60)

from evalplus.data import get_mbpp_plus
from evalplus.eval import PASS

t0 = time.time()
mbppp_result = None

try:
    # NOTE: evalplus's MBPP+ is a separate, sanitized ~399-problem subset with its
    # own task_ids and prompts — it does NOT line up 1:1 with the raw 974-problem
    # MBPP test split used in the cell above, so we can't reuse those completions.
    # We generate fresh (but only for this smaller set, not all 974 again).
    mbppp_problems = get_mbpp_plus()
    mbppp_task_ids = list(mbppp_problems.keys())
    n_tasks = len(mbppp_task_ids)
    print(f"  evalplus MBPP+ has {n_tasks} sanitized problems (independent of the raw MBPP split above)")

    samples_path = f"{OUTPUT_DIR}/mbpp_plus_samples.jsonl"
    result_path = samples_path.replace(".jsonl", "_eval_results.json")
    for p in (samples_path, result_path):
        if os.path.exists(p):
            os.remove(p)

    with open(samples_path, "w") as f:
        for i, task_id in enumerate(mbppp_task_ids):
            prompt = mbppp_problems[task_id]["prompt"]
            try:
                gen = generate_one(prompt)
                code = extract_code(gen)
            except Exception:
                code = ""
            f.write(json.dumps({"task_id": task_id, "solution": code}) + "\n")
            if (i + 1) % 50 == 0 or i == n_tasks - 1:
                print(f"  Generated {i+1}/{n_tasks} solutions ...")

    from evalplus.evaluate import evaluate as ev_eval
    ev_eval(dataset="mbpp", samples=samples_path, i_just_wanna_run=True)

    with open(result_path) as f:
        raw = json.load(f)
    n = len(raw["eval"])
    base_pass = sum(1 for r in raw["eval"].values() if r[0]["base_status"] == PASS)
    plus_pass = sum(1 for r in raw["eval"].values()
                     if r[0]["base_status"] == PASS and r[0]["plus_status"] == PASS)
    mbppp_result = {
        "benchmark": "MBPP+", "total": n, "passed": plus_pass,
        "pass@1": round(plus_pass / n * 100, 1),
        "base_pass@1": round(base_pass / n * 100, 1),
        "evalplus_raw_path": result_path,
    }
    print(f"  MBPP+ pass@1 = {mbppp_result['pass@1']}%  (base-only pass@1 = {mbppp_result['base_pass@1']}%)")

except Exception as e:
    print(f"[mbpp+] evalplus failed: {e}")
    traceback.print_exc()
    mbppp_result = {**mbpp_result, "benchmark": "MBPP+", "note": f"evalplus error: {e}"}

print(f"  Done in {(time.time()-t0)/60:.1f} min")

In [ ]:
BASELINES = {
    "Model": ["TIMPS-Coder-7B (this model)", "Qwen2.5-Coder-7B-Instruct", "Qwen2.5-Coder-7B",
              "DeepSeek-Coder-7B-Instruct-v1.5", "CodeLlama-7B-Instruct", "CodeGemma-7B-it",
              "StarCoder2-7B", "Llama-3.1-8B-Instruct", "Phi-3.5-mini-instruct (3.8B)", "Gemma-2-9B-it"],
    "HumanEval":  ["—", "86.6", "89.6", "84.1", "53.7", "56.1", "40.2", "72.6", "68.8", "54.3"],
    "HumanEval+": ["—", "71.3", "76.2", "70.8", "44.5", "46.9", "32.9", "61.0", "57.9", "44.5"],
    "MBPP":       ["—", "82.0", "84.0", "79.6", "55.6", "61.8", "46.0", "70.8", "73.0", "59.6"],
    "MBPP+":      ["—", "69.6", "72.0", "68.4", "45.0", "50.6", "36.5", "58.7", "61.3", "49.3"],
    "Params":     ["7B", "7.6B", "7.6B", "7.1B", "6.7B", "7.0B", "7.0B", "8.0B", "3.8B", "9.2B"],
}

results = {"humaneval": he_result, "humaneval_plus": hep_result, "mbpp": mbpp_result, "mbpp_plus": mbppp_result}

# Fill in TIMPS-Coder scores
bl = {k: list(v) for k, v in BASELINES.items()}
bl["HumanEval"][0]  = str(he_result.get("pass@1", "—"))
bl["HumanEval+"][0] = str(hep_result.get("pass@1", "—"))
bl["MBPP"][0]       = str(mbpp_result.get("pass@1", "—"))
bl["MBPP+"][0]      = str(mbppp_result.get("pass@1", "—"))

cols = list(bl.keys())
hdr = "| " + " | ".join(cols) + " |"
sep = "|" + "|".join(["---"] * len(cols)) + "|"
rows = ["| " + " | ".join(str(bl[c][i]) for c in cols) + " |" for i in range(len(bl["Model"]))]
table = "\n".join([hdr, sep] + rows)

print(table)

# Save results
payload = {
    "model": MODEL_ID, "timestamp": datetime.now(timezone.utc).isoformat(),
    "results_summary": {name: {"pass@1": r.get("pass@1"), "total": r.get("total"), "passed": r.get("passed")} for name, r in results.items()},
    "detailed_results": results,
}
with open(f"{OUTPUT_DIR}/benchmark_results.json", "w") as f:
    json.dump(payload, f, indent=2, default=str)
print(f"\nResults saved to {OUTPUT_DIR}/benchmark_results.json")

In [ ]:
he  = he_result.get("pass@1", "—")
hep = hep_result.get("pass@1", "—")
mb  = mbpp_result.get("pass@1", "—")
mbp = mbppp_result.get("pass@1", "—")

MODEL_CARD = f"""---
license: apache-2.0
language: [en]
base_model: Qwen/Qwen2.5-Coder-7B-Instruct
tags: [code, qwen2.5, lora, merged, sft, dpo, grpo]
library_name: transformers
pipeline_tag: text-generation
model-index:
- name: TIMPS-Coder-7B
  results:
  - task: {{"type": "text-generation", "name": "Code Generation"}}
    dataset: {{"type": "openai_humaneval", "name": "HumanEval"}}
    metrics:
    - {{type: "pass@1", "value": {he}, "name": "pass@1"}}
  - task: {{"type": "text-generation", "name": "Code Generation"}}
    dataset: {{"type": "evalplus/humanevalplus", "name": "HumanEval+"}}
    metrics:
    - {{type: "pass@1", "value": {hep}, "name": "pass@1"}}
  - task: {{"type": "text-generation", "name": "Code Generation"}}
    dataset: {{"type": "mbpp", "name": "MBPP"}}
    metrics:
    - {{type: "pass@1", "value": {mb}, "name": "pass@1"}}
  - task: {{"type": "text-generation", "name": "Code Generation"}}
    dataset: {{"type": "evalplus/mbppplus", "name": "MBPP+"}}
    metrics:
    - {{type: "pass@1", "value": {mbp}, "name": "pass@1"}}
---

# TIMPS-Coder-7B

TIMPS-Coder-7B is a code-generation model built by fine-tuning **Qwen2.5-Coder-7B-Instruct**
through a 4-step pipeline: SFT, GRPO, DPO, and Self-Guided Self-Play.

## Benchmark Results

| Benchmark | Score |
|-----------|-------|
| **HumanEval pass@1** | **{he}%** |
| **HumanEval+ pass@1** | **{hep}%** |
| **MBPP pass@1** | **{mb}%** |
| **MBPP+ pass@1** | **{mbp}%** |

### Comparison with 7B-9B Code Models

{table}

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("sandeeprdy1729/TIMPS-Coder-7B", device_map="auto", torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained("sandeeprdy1729/TIMPS-Coder-7B")
messages = [{{"role": "user", "content": "Write a fibonacci function."}}]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
print(tokenizer.decode(model.generate(inputs, max_new_tokens=512)[0]))
"""

with open(f"{OUTPUT_DIR}/model_card_results.md", "w") as f:
    f.write(MODEL_CARD)
print(f"Model card saved to {OUTPUT_DIR}/model_card_results.md")
print(MODEL_CARD)

In [ ]:

#**Cell 11 — Push scores to HuggingFace model card (optional):**
# ```python
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj=f"{OUTPUT_DIR}/model_card_results.md",
    path_in_repo="README.md",
    repo_id=MODEL_ID,
    repo_type="model",
    commit_message=f"Add benchmark results: HE={he}% HE+={hep}% MBPP={mb}% MBPP+={mbp}%",
)
print(f"Model card updated at https://huggingface.co/{MODEL_ID}")